# Week 6 — Neural Decoding: Can Population Activity Predict Experimental Condition?

**Supervised learning, validation design, and data leakage in population neural data**

A common goal in computational neuroscience is to ask whether information about an experimental condition can be decoded from neural activity.

The scientific question for this notebook is:

> **Can population neural responses predict whether an animal belongs to the control or stimulated condition?**

The dataset is synthetic but structured like an experiment in which:

- 14 animals contribute repeated neural-response trials;
- 30 neurons are measured on every trial;
- treatment is assigned at the **animal level**;
- neural activity varies across trials and across animals.

## Learning goals

By the end of class you should be able to:

1. define features, labels, observations, and prediction targets in a neural decoding problem;
2. explain the basic logic of logistic regression;
3. distinguish training performance from out-of-sample performance;
4. explain why a random trial split can create data leakage when treatment is assigned by animal;
5. evaluate a classifier using held-out animals;
6. use a permutation control to define a chance-level reference;
7. interpret classifier weights cautiously.


In [ ]:
from io import BytesIO
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

pd.set_option('display.max_columns', 40)

GITHUB_RAW = (
    'https://raw.githubusercontent.com/'
    'willi3by/Survey_of_Methods_in_Comp_Neuro_Fall_2026/'
    'main/Data/Week_06'
)

def github_csv(filename):
    url = f"{GITHUB_RAW}/{filename}"
    req = Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    try:
        with urlopen(req) as response:
            return pd.read_csv(BytesIO(response.read()))
    except HTTPError as e:
        raise RuntimeError(
            f"GitHub returned HTTP {e.code} for {url}. "
            "Check that the file exists in Data/Week_06 on the main branch."
        ) from e
    except URLError as e:
        raise RuntimeError(
            f"Could not reach GitHub while loading {url}. "
            "Check the notebook's internet connection."
        ) from e

activity = github_csv('population_activity.csv')
animals = github_csv('animal_metadata.csv')

data = activity.merge(animals, on='animal_id', validate='many_to_one')

neuron_cols = [c for c in data.columns if c.startswith('neuron_')]

print(f"{len(data):,} trials")
print(f"{data['animal_id'].nunique()} animals")
print(f"{len(neuron_cols)} neural features")
data.head()


## 1. What is a neural decoder?

A **decoder** is a model that uses neural measurements to predict some variable of interest.

For this problem:

- one **observation** = one trial;
- the **features** = activity of 30 neurons;
- the **label** = control or stimulated;
- the **biological unit receiving treatment** = animal.


The classifier receives the neural vector \(\mathbf{x}_{ij}\) and tries to predict the treatment label \(T_i\).

### Interpretation prompt

Before fitting a model:

1. How many rows are in the dataset?
2. How many independent treatment assignments occurred?
3. Are 1,120 trials equivalent to 1,120 independently treated experimental units?


In [ ]:
summary = (
    data.groupby(['animal_id', 'treatment'])
        .size()
        .rename('n_trials')
        .reset_index()
)
summary


## 2. Look at the population activity

Each trial is a point in a 30-dimensional neural feature space.

A useful first view is to average repeated trials within each animal and display the mean response of every recorded neuron.

This visualization can reveal two different sources of structure:

- **condition-related structure** shared across animals in the same group;
- **animal-specific structure** that is stable across repeated trials from one animal.

A classifier can exploit either source if the validation design allows it.


In [ ]:
animal_means = (
    data.groupby(['animal_id', 'treatment'])[neuron_cols]
        .mean()
        .reset_index()
)

matrix = animal_means[neuron_cols].to_numpy()

fig, ax = plt.subplots(figsize=(11, 5))
im = ax.imshow(matrix, aspect='auto', cmap='viridis')

ax.set_yticks(np.arange(len(animal_means)))
ax.set_yticklabels(
    animal_means['animal_id'] + ' (' + animal_means['treatment'].str[0] + ')'
)
ax.set_xlabel('Neuron')
ax.set_ylabel('Animal')
ax.set_title('Mean population response for each animal')
plt.colorbar(im, ax=ax, label='Mean response')
plt.show()


## 3. Logistic regression as a decoder

For a binary outcome, logistic regression models the probability of one class.

Let $T_i = 1$ denote the stimulated condition and $T_i = 0$ denote control.

The model first computes a weighted sum of neural activity:

$$
z_{ij}
=
\beta_0
+
\beta_1 x_{ij1}
+
\beta_2 x_{ij2}
+
\cdots
+
\beta_p x_{ijp}
$$

It then converts that value into a probability using the logistic function:

$$
P(T_i = 1 \mid \mathbf{x}_{ij})
=
\frac{1}{1 + e^{-z_{ij}}}
$$

If the predicted probability is above a decision threshold, the trial is classified as stimulated.

### Why standardize the neural features?

Neurons can have different response ranges. Standardization puts them on comparable scales:

$$
z = \frac{x - \mu}{\sigma}
$$

The scaler must be estimated using the **training data only**. A pipeline handles that step correctly during fitting and prediction.

In [ ]:
X = data[neuron_cols].to_numpy()
y = data['treatment_binary'].to_numpy()

decoder = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=3000)
)

X.shape, y.shape


## 4. A tempting approach: randomly split trials

A standard machine-learning workflow randomly assigns observations to training and test sets.

The model is fit using the training observations and evaluated on observations it did not directly see during fitting.

That sounds appropriate, but the definition of an "unseen observation" depends on the scientific design.

Run the random trial split below and examine the accuracy.


In [ ]:
train_idx, test_idx = train_test_split(
    np.arange(len(data)),
    test_size=0.30,
    random_state=42,
    stratify=y
)

decoder.fit(X[train_idx], y[train_idx])

pred_random = decoder.predict(X[test_idx])
acc_random = accuracy_score(y[test_idx], pred_random)

print(f"Random trial-split accuracy: {acc_random:.3f}")


In [ ]:
cm = confusion_matrix(y[test_idx], pred_random)

fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(cm, cmap='Blues')

ax.set_xticks([0, 1], labels=['Control', 'Stimulated'])
ax.set_yticks([0, 1], labels=['Control', 'Stimulated'])
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
ax.set_title('Random trial split')

for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center')

plt.colorbar(im, ax=ax, label='Trials')
plt.show()


The random split may produce impressive accuracy.

Before concluding that treatment can be decoded reliably, ask:

1. Can trials from the same animal appear in both the training and test sets?
2. Is treatment assigned separately on every trial?
3. Could stable animal-specific response patterns help identify the animal?
4. If the model recognizes an animal, does it indirectly know that animal's treatment label?

This is a form of **data leakage across biological units**.

The test trials are new trials, but they are not necessarily new animals.


In [ ]:
train_animals = set(data.iloc[train_idx]['animal_id'])
test_animals = set(data.iloc[test_idx]['animal_id'])

overlap = sorted(train_animals.intersection(test_animals))

print(f"Animals represented in training set: {len(train_animals)}")
print(f"Animals represented in test set: {len(test_animals)}")
print(f"Animals appearing in BOTH sets: {len(overlap)}")
print(overlap)


## 5. Validate on animals the model has never seen

The prediction question should determine the validation scheme.

If the goal is:

> **Can this neural signature generalize to a new animal?**

then all trials from a held-out animal must remain outside the training data.

The experiment contains seven matched animal pairs. Each pair contains:

- one control animal;
- one stimulated animal.

For each held-out pair we:

1. fit the entire preprocessing-and-classification pipeline on the other 12 animals;
2. predict the two animals in the held-out pair;
3. compute accuracy using only those held-out trials.


In [ ]:
pair_scores = []

for pair_id in sorted(data['pair_id'].unique()):
    train = data['pair_id'].to_numpy() != pair_id
    test = ~train

    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=3000)
    )

    model.fit(X[train], y[train])
    pred = model.predict(X[test])

    pair_scores.append({
        'pair_id': int(pair_id),
        'held_out_animals': ', '.join(sorted(data.loc[test, 'animal_id'].unique())),
        'accuracy': accuracy_score(y[test], pred)
    })

pair_results = pd.DataFrame(pair_scores)
pair_results


In [ ]:
animal_cv_accuracy = pair_results['accuracy'].mean()

comparison = pd.DataFrame({
    'validation_scheme': [
        'Random trial split',
        'Held-out animals'
    ],
    'accuracy': [
        acc_random,
        animal_cv_accuracy
    ]
})

comparison


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

ax.bar(
    comparison['validation_scheme'],
    comparison['accuracy']
)

ax.axhline(0.5, linestyle='--', linewidth=1.5, label='Chance for balanced classes')
ax.set_ylim(0, 1)
ax.set_ylabel('Accuracy')
ax.set_title('Validation design changes the conclusion')
ax.legend()
plt.xticks(rotation=12)
plt.tight_layout()
plt.show()


### What changed?

The neural measurements did not change.

The classifier did not change.

The outcome label did not change.

Only the **definition of independent test data** changed.

A large gap between the two estimates indicates that a random trial split was benefiting from information that does not generalize cleanly to new animals.

This distinction is central to neural decoding:

> A model can predict held-out measurements very well without necessarily generalizing to held-out biological subjects.


## 6. Define chance with a permutation control

For balanced binary classes, 50% is a simple theoretical chance level. A permutation analysis gives a dataset-specific reference.

Treatment is assigned at the **animal level**, and the experimental design contains matched animal pairs. Therefore the permutation should preserve that design.

For each permutation:

1. randomly swap the control/stimulated labels within each matched pair;
2. assign the shuffled animal label to every trial from that animal;
3. repeat the held-out-pair validation;
4. save the mean accuracy.

This creates a null distribution in which treatment labels are unrelated to neural activity while preserving the number of animals, repeated trials, pair structure, and class balance.

A trial-level label permutation would create biologically impossible datasets in which different trials from the same animal receive different treatment assignments.


In [ ]:
rng = np.random.default_rng(7)

animal_table = (
    data[['animal_id', 'treatment_binary', 'pair_id']]
    .drop_duplicates()
    .sort_values(['pair_id', 'animal_id'])
    .reset_index(drop=True)
)

n_permutations = 100
permuted_scores = []

for _ in range(n_permutations):
    perm_map = {}

    # Randomly swap the two treatment labels within each matched pair.
    for pair_id, pair in animal_table.groupby('pair_id'):
        shuffled = pair['treatment_binary'].to_numpy().copy()
        rng.shuffle(shuffled)
        perm_map.update(dict(zip(pair['animal_id'], shuffled)))

    y_perm = data['animal_id'].map(perm_map).to_numpy()

    pair_acc = []

    for pair_id in sorted(data['pair_id'].unique()):
        train = data['pair_id'].to_numpy() != pair_id
        test = ~train

        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=3000)
        )

        model.fit(X[train], y_perm[train])
        pred = model.predict(X[test])
        pair_acc.append(accuracy_score(y_perm[test], pred))

    permuted_scores.append(np.mean(pair_acc))

permuted_scores = np.asarray(permuted_scores)

print(f"Observed held-out-animal accuracy: {animal_cv_accuracy:.3f}")
print(f"Mean permuted accuracy: {permuted_scores.mean():.3f}")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

ax.hist(permuted_scores, bins=16, edgecolor='black', alpha=0.75)
ax.axvline(
    animal_cv_accuracy,
    linewidth=2,
    label=f'Observed = {animal_cv_accuracy:.2f}'
)

ax.set_xlabel('Held-out-animal accuracy')
ax.set_ylabel('Permutation count')
ax.set_title('Animal-level permutation control')
ax.legend()
plt.show()


## 7. What does the classifier use?

A logistic-regression model assigns a coefficient to each neural feature.

Because the features are standardized inside the pipeline:

- a positive coefficient pushes predictions toward the stimulated class;
- a negative coefficient pushes predictions toward the control class;
- a larger absolute coefficient means the feature has more influence on the fitted decision boundary.

Fit one model using the complete dataset and inspect the standardized coefficients.


In [ ]:
final_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=3000)
)

final_model.fit(X, y)

weights = final_model.named_steps['logisticregression'].coef_[0]

weight_table = pd.DataFrame({
    'neuron': neuron_cols,
    'weight': weights,
    'abs_weight': np.abs(weights)
}).sort_values('abs_weight', ascending=False)

weight_table.head(10)


In [ ]:
top = weight_table.head(12).sort_values('weight')

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(top['neuron'], top['weight'])
ax.axvline(0, linewidth=1)
ax.set_xlabel('Standardized logistic-regression weight')
ax.set_ylabel('Neural feature')
ax.set_title('Largest classifier weights')
plt.tight_layout()
plt.show()


### Why a large classifier weight is not automatically a biological mechanism

Classifier coefficients answer a predictive question:

> Which weighted combination of features best separates the classes in this dataset?

They do **not** automatically tell us:

- which neuron was causally altered by treatment;
- which neuron is independently informative;
- which neuron would retain the same weight in another sample;
- whether correlated neurons are interchangeable in the model.

When predictors are correlated, several different coefficient patterns can produce similar predictions.

Prediction and biological explanation are related, but they are not the same task.
